# Team Mapping

## Get all CBBD teams

In [1]:
from cbb_predictor.db import get_connection

with get_connection(read_only=True) as con:
    print(con.sql("SELECT COUNT(*), MIN(FetchedAtUTC), MAX(FetchedAtUTC) FROM cbbd_games").fetchall())

[(6006, datetime.datetime(2026, 10, 31, 20, 4, 38, 957039), datetime.datetime(2026, 10, 31, 20, 4, 38, 957039))]


In [2]:
with get_connection(read_only=True) as con:
    cbbd_teams = con.sql("""
        SELECT homeTeamId AS cbbd_id, homeTeam AS cbbd_name FROM cbbd_games
        UNION
        SELECT awayTeamId, awayTeam FROM cbbd_games
    """).df()
    
    games = con.sql("SELECT homeTeamId, awayTeamId FROM cbbd_games").df()

print(len(cbbd_teams), cbbd_teams["cbbd_id"].is_unique)


748 True


## Normalize names

In [3]:
import unicodedata

def normalize_name(name: str) -> str:
    """Lowercase, trim, and strip accents so names can be compared."""
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")
    return name.lower().strip()

normalize_name("San José State ")   # 'san jose state'

'san jose state'

In [4]:
cbbd_teams["normalized"] = cbbd_teams["cbbd_name"].map(normalize_name)
cbbd_teams.head(10)

,cbbd_id,cbbd_name,normalized
0,269,South Dakota,south dakota
1,241,Rice,rice
2,336,Vanderbilt,vanderbilt
3,158,Marshall,marshall
4,188,Nebraska,nebraska
5,282,Stephen F. Austin,stephen f. austin
6,23,Bethune-Cookman,bethune-cookman
7,287,TCU,tcu
8,252,Saint Louis,saint louis
9,220,Ole Miss,ole miss


## Checks

In [5]:
cbbd_teams[cbbd_teams["cbbd_name"].str.lower() != cbbd_teams["normalized"]]

,cbbd_id,cbbd_name,normalized
244,260,San José State,san jose state


## Map cbbd to kaggle

In [6]:
import pandas as pd
from cbb_predictor.config import RAW_DIR

spellings = pd.read_csv(RAW_DIR / "MTeamSpellings.csv")
kaggle_teams = pd.read_csv(RAW_DIR / "MTeams.csv")

mapped = cbbd_teams.merge(
    spellings, left_on="normalized", right_on="TeamNameSpelling",
    how="left", validate="many_to_one",
)
print(mapped["TeamID"].notna().sum(), "matched,", mapped["TeamID"].isna().sum(), "unmatched")

362 matched, 386 unmatched


In [7]:
CURRENT_SEASON = 2027

# Renamed D1 teams: normalized CBBD name -> Kaggle TeamID
TEAM_OVERRIDES = {
    "ualbany": 1107,
    "app state": 1111,
    "miami": 1274,               # CBBD's plain "Miami" is Miami (FL); Miami (OH) is listed separately
    "ut rio grande valley": 1410,
    "ul monroe": 1419,
    "st. thomas-minnesota": 1472,
    "queens university": 1474,
}

# Teams new to D1 this season, not in Kaggle: given IDs above Kaggle's range
NEW_TEAMS = {
    "west florida": 1600,
}

current_d1 = kaggle_teams.loc[kaggle_teams["LastD1Season"] >= CURRENT_SEASON - 1, "TeamID"]

# 1. Match against Kaggle's spellings
mapped = cbbd_teams.merge(
    spellings, left_on="normalized", right_on="TeamNameSpelling",
    how="left", validate="many_to_one",
).drop(columns="TeamNameSpelling")

# 2. discard matches to teams that are no longer D1
mapped.loc[~mapped["TeamID"].isin(current_d1), "TeamID"] = pd.NA

# 3. fill gaps with manual overrides
mapped["TeamID"] = mapped["TeamID"].fillna(mapped["normalized"].map(TEAM_OVERRIDES))

# 4. assign IDs to new D1 teams
mapped["TeamID"] = mapped["TeamID"].fillna(mapped["normalized"].map(NEW_TEAMS))

mapped["TeamID"] = mapped["TeamID"].astype("Int64")

## Check

In [8]:
d1 = mapped.dropna(subset=["TeamID"])

# All current D1 teams mapped (St. Francis PA, 1384, is the expected exception)
print(sorted(set(current_d1) - set(d1["TeamID"])))          # [1384]

# No TeamID assigned to two CBBD teams
print(d1[d1["TeamID"].duplicated(keep=False)])              # empty

# Games where both teams are D1
d1_ids = set(d1["cbbd_id"])
both_d1 = games["homeTeamId"].isin(d1_ids) & games["awayTeamId"].isin(d1_ids)
print(both_d1.sum(), "kept,", (~both_d1).sum(), "dropped")

[1384]
Empty DataFrame
Columns: [cbbd_id, cbbd_name, normalized, TeamID]
Index: []
5413 kept, 593 dropped
